# 08 — How IRB Risk Parameters Would Be Estimated

## From synthetic assumptions to real estimates

The project uses synthetic PD, LGD and own CCF values so the complete capital calculation can be demonstrated. This notebook shows the data and estimation route a bank would need before those parameters could be treated as internal estimates.

This step separates parameter estimation from parameter use. The following notebooks apply the synthetic values at loan level; they do not claim that a production model has been fitted.

### Estimation language used below

| Term | Simple meaning |
|---|---|
| Observation window | the historical period used to estimate a parameter |
| Reference date | the date from which a one-year default outcome or pre-default drawdown is measured |
| Margin of conservatism | an adjustment for data or estimation uncertainty |
| Downturn | a period in which defaults or credit losses are materially above normal |

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display as notebook_display
from pandas.io.formats.style import Styler

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from basel_credit_risk.config import load_all_config

CRORE = 10_000_000
COLORS = ["#0B3A53", "#1F77B4", "#2A9D8F", "#E9C46A", "#F4A261", "#E76F51"]
sns.set_theme(style="whitegrid", context="notebook")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)
pd.set_option("display.expand_frame_repr", False)


def display(*objects, **kwargs):
    # Show complete table text and wrap long descriptions in saved notebook output.
    wrapped_objects = []
    for displayed_object in objects:
        if isinstance(displayed_object, pd.DataFrame):
            displayed_object = displayed_object.style
        if isinstance(displayed_object, Styler):
            displayed_object = displayed_object.set_properties(**{
                "white-space": "normal",
                "overflow-wrap": "anywhere",
                "text-align": "left",
                "vertical-align": "top",
                "max-width": "420px",
            }).set_table_styles([
                {"selector": "table", "props": [("width", "100%"), ("table-layout", "auto")]},
                {"selector": "th", "props": [
                    ("white-space", "normal"),
                    ("overflow-wrap", "anywhere"),
                    ("word-break", "break-word"),
                    ("text-align", "left"),
                    ("vertical-align", "top"),
                    ("max-width", "180px"),
                ]},
            ], overwrite=False)
        wrapped_objects.append(displayed_object)
    return notebook_display(*wrapped_objects, **kwargs)

configs = load_all_config(ROOT / "config")
raw = pd.read_csv(ROOT / "data/generated/current_portfolio.csv.gz", low_memory=False)
loans = pd.read_csv(ROOT / "data/processed/current_calculated.csv.gz", low_memory=False)

print(f"Portfolio represented below: {len(loans):,} synthetic loan-level exposures")

Portfolio represented below: 30,000 synthetic loan-level exposures


## What is synthetic and what a bank would calculate

The values used in this project are visible assumptions. A bank would replace them with estimates built from governed historical data, documented definitions, independent validation and supervisory approval where required.

In [2]:
estimation_map = pd.DataFrame([
    ["Long-run PD", "Synthetic transition-matrix default column", "Borrowers in each grade at annual reference dates; defaults during the next 12 months; migrations; overrides; cured defaults", "Calculate annual one-year default rates by grade or pool, cover representative good and bad years, then form a long-run average and add conservatism"],
    ["Downturn LGD", "Synthetic recoveries, costs and recovery times", "Defaulted facilities; EAD at default; dated cash recoveries; collateral proceeds; direct and indirect workout costs", "Calculate discounted economic loss for each default, estimate a long-run default-weighted LGD, then reflect periods of high loss severity"],
    ["IRB CCF / EAD", "Synthetic own CCF by facility", "Drawn and undrawn amounts at fixed pre-default dates; limits; cancellations; drawings at default", "Calculate realised conversion of unused limits, estimate long-run default-weighted EAD or CCF and reflect downturn dependence where material"],
    ["Effective maturity", "Contractual maturity with represented Basel bounds", "Cash-flow schedule, repricing and contractual maturity", "Calculate the applicable effective maturity under the relevant rule, including permitted floors, caps and exemptions"],
], columns=["Parameter", "Value used here", "Data a bank needs", "Ideal estimation route"])
display(estimation_map)

,Parameter,Value used here,Data a bank needs,Ideal estimation route
0,Long-run PD,Synthetic transition-matrix default column,Borrowers in each grade at annual reference dates; defaults during the next 12 months; migrations; overrides; cured defaults,"Calculate annual one-year default rates by grade or pool, cover representative good and bad years, then form a long-run average and add conservatism"
1,Downturn LGD,"Synthetic recoveries, costs and recovery times",Defaulted facilities; EAD at default; dated cash recoveries; collateral proceeds; direct and indirect workout costs,"Calculate discounted economic loss for each default, estimate a long-run default-weighted LGD, then reflect periods of high loss severity"
2,IRB CCF / EAD,Synthetic own CCF by facility,Drawn and undrawn amounts at fixed pre-default dates; limits; cancellations; drawings at default,"Calculate realised conversion of unused limits, estimate long-run default-weighted EAD or CCF and reflect downturn dependence where material"
3,Effective maturity,Contractual maturity with represented Basel bounds,"Cash-flow schedule, repricing and contractual maturity","Calculate the applicable effective maturity under the relevant rule, including permitted floors, caps and exemptions"


The first row is primarily a borrower or retail-pool estimate. LGD and EAD are facility estimates because collateral, seniority and undrawn commitments can differ even for the same borrower.

## Small PD example

Suppose a grade contains 10,000 borrowers at the start of each year. The observed one-year default rate is the number that defaults within the following year divided by the starting borrower count. The example averages across years only to show the arithmetic; a bank would also test representativeness, grade philosophy, overrides, missing years and uncertainty.

In [3]:
pd_history = pd.DataFrame({
    "Year": [2021, 2022, 2023, 2024, 2025],
    "Borrowers at start": [10000, 10400, 10800, 11100, 11400],
    "Defaults within one year": [70, 83, 162, 122, 91],
})
pd_history["Observed one-year default rate"] = pd_history["Defaults within one year"] / pd_history["Borrowers at start"]
long_run_example = pd_history["Defaults within one year"].sum() / pd_history["Borrowers at start"].sum()
display(pd_history.style.format({"Observed one-year default rate": "{:.2%}"}))
print(f"Count-weighted long-run example: {long_run_example:.2%}")

,Year,Borrowers at start,Defaults within one year,Observed one-year default rate
0,2021,10000,70,0.70%
1,2022,10400,83,0.80%
2,2023,10800,162,1.50%
3,2024,11100,122,1.10%
4,2025,11400,91,0.80%


Count-weighted long-run example: 0.98%


## Small LGD and CCF examples

Economic LGD uses discounted recoveries and workout costs. Realised CCF measures how much of the unused commitment was drawn before default. These two examples show the core arithmetic, not a fitted model.

In [4]:
examples = pd.DataFrame({
    "Calculation": ["LGD", "Realised CCF"],
    "Inputs": ["EAD 1.00; discounted recoveries 0.58; workout costs 0.07", "Drawn at reference 0.40; drawn at default 0.70; undrawn at reference 0.60"],
    "Formula": ["(1.00 - 0.58 + 0.07) / 1.00", "(0.70 - 0.40) / 0.60"],
    "Result": [(1.00 - 0.58 + 0.07) / 1.00, (0.70 - 0.40) / 0.60],
})
display(examples.style.format({"Result": "{:.1%}"}))

,Calculation,Inputs,Formula,Result
0,LGD,EAD 1.00; discounted recoveries 0.58; workout costs 0.07,(1.00 - 0.58 + 0.07) / 1.00,49.0%
1,Realised CCF,Drawn at reference 0.40; drawn at default 0.70; undrawn at reference 0.60,(0.70 - 0.40) / 0.60,50.0%


The project now moves from estimation design to the synthetic parameter values actually used in the calculation. Their source remains visible on each loan row.

The estimation route is documented separately from the synthetic values used to demonstrate the IRB calculation.